# COSMOS posterior bootstrap versus spec-z broadband flow

Run the three scripts from the repository root before executing this notebook:
```bash
python bin/lsst_like.py
python bin/match_cosmos.py
python bin/train_flows.py
```
Both models use `data/parent.npz`. Only the shared held-out secure-spec sample is
scored; this sample is spectroscopically selected and does not establish performance
for the entire magnitude-limited population. COSMOS PDFs are assumed unbiased.
The fluxes use native CFHT/HSC filters and an approximate LSST noise forecast.

Figures and executed notebooks stay local under `results/`. Earlier paper analyses
and notebooks are archived locally; this notebook describes the simplified workflow.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'setup.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pzflow.utils import coverage

os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
plt.rcParams.update(
    {
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 13,
        'axes.linewidth': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'legend.frameon': False,
    }
)
OUT = ROOT / 'results/flows'
run = json.loads((OUT / 'run.json').read_text())
with Path(run['parent']).open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == run['parent_sha256']
pred = dict(np.load(OUT / 'predictions.npz'))
with np.load(run['parent']) as parent:
    test = (parent['split'] == 2) & parent['has_specz']
    assert np.array_equal(parent['id'][test], pred['id'])
    assert np.array_equal(parent['flux'][test], pred['flux'])
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)
models = [('bootstrapped', 'COSMOS PDF bootstrap', 'C0'), ('specz', 'Spec-z broadband flow', 'C1')]
z, mag = pred['zspec'], pred['imag']
run['models']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (key, label, color) in zip(axes, models):
    ax.scatter(z, pred[key + '_quantiles'][:, 2], s=3, alpha=0.2, color=color, rasterized=True)
    ax.plot([0, 6], [0, 6], 'k--', lw=1)
    ax.set(xlabel=r'$z_{\rm spec}$', title=label, xlim=(0, 6), ylim=(0, 6))
axes[0].set_ylabel(r'$z_{\rm phot}$ (posterior median)')
fig.tight_layout()
fig.savefig(FIG / 'accuracy.png', dpi=180)

In [ ]:
levels = np.linspace(0, 1, 101)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for key, label, color in models:
    axes[0].plot(levels, coverage(pred[key + '_pit'], levels), label=label, color=color)
    axes[1].hist(
        pred[key + '_pit'], bins=20, density=True, histtype='step', color=color, label=label
    )
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[0].set(
    xlabel='Nominal central probability', ylabel='Empirical coverage', xlim=(0, 1), ylim=(0, 1)
)
axes[1].set(xlabel='PIT', ylabel='Probability density', xlim=(0, 1))
axes[0].legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG / 'calibration.png', dpi=180)

Magnitude bins use native HSC i aperture magnitude, with each point at the
median magnitude of the same galaxies used for scoring. Counts are printed above
the plot. Shading gives 95% intervals from 200 paired galaxy bootstrap resamples
within each bin, with `alpha=0.5`. These intervals describe evaluation-sample
variation; they do not include variation from retraining or cosmic variance.

In [ ]:
edges = np.array([18.0, 22.5, 23.75, 24.5, 25.0])
centers, counts = [], []
values = {key: [] for key, _, _ in models}
bounds = {key: [] for key, _, _ in models}
rng = np.random.default_rng(20261027)


def statistic(key, idx):
    residual = (pred[key + '_quantiles'][idx, 2] - z[idx]) / (1 + z[idx])
    nmad = 1.4826 * np.median(abs(residual - np.median(residual)))
    return np.array([nmad, coverage(pred[key + '_pit'][idx], [0.68])[0]])


for lo, hi in zip(edges[:-1], edges[1:]):
    idx = np.flatnonzero((mag >= lo) & (mag < hi))
    if not len(idx):
        continue
    centers.append(np.median(mag[idx]))
    counts.append(len(idx))
    draws = rng.choice(idx, size=(200, len(idx)), replace=True)
    for key, _, _ in models:
        values[key].append(statistic(key, idx))
        boot = np.array([statistic(key, draw) for draw in draws])
        bounds[key].append(np.quantile(boot, [0.025, 0.975], axis=0))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for key, label, color in models:
    v, b = np.asarray(values[key]), np.asarray(bounds[key])
    for j, ax in enumerate(axes):
        ax.plot(centers, v[:, j], 'o-', color=color, label=label)
        ax.fill_between(centers, b[:, 0, j], b[:, 1, j], color=color, alpha=0.5)
for ax in axes:
    for x, count in zip(centers, counts):
        ax.text(
            x, 1.02, f'$N={count}$', transform=ax.get_xaxis_transform(), ha='center', fontsize=10
        )
    ax.set_xlabel('HSC i aperture magnitude [AB]')
axes[0].set_ylabel(r'$\sigma_{\rm NMAD}$')
axes[1].set_ylabel('68 percent interval coverage')
axes[1].axhline(0.68, color='k', ls='--', lw=1)
axes[0].legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG / 'magnitude.png', dpi=180)